In [2]:
import xarray as xr
import fsspec
import zarr
import dask.array as da
import numpy as np
from dask.diagnostics import ProgressBar

In [2]:
fs = fsspec.filesystem("gs", token="anon")

# Dataset root (for coords)
root = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
store_root = fs.get_mapper(root)

# Array paths (for fast open of only u/v metadata)
rootU = root + "/u_component_of_wind"
rootV = root + "/v_component_of_wind"
storeU = fs.get_mapper(rootU)
storeV = fs.get_mapper(rootV)

# Open only the array metadata you care about
u_z = zarr.open_array(storeU, mode="r")
v_z = zarr.open_array(storeV, mode="r")

# Lazy dask arrays (no data read yet)
u_d = da.from_zarr(u_z)
v_d = da.from_zarr(v_z)

# Dimension names written by xarray into zarr attrs
dims = tuple(u_z.attrs["_ARRAY_DIMENSIONS"])

# Load coordinate arrays from the dataset root (small; ok to read)
# (Only include coords that actually exist as arrays at the root.)
root_group = zarr.open_group(store_root, mode="r")
coords = {d: root_group[d][:] for d in dims if d in root_group}

u = xr.DataArray(u_d, dims=dims, coords=coords, name="u_component_of_wind")
v = xr.DataArray(v_d, dims=dims, coords=coords, name="v_component_of_wind")

uv = xr.Dataset({"u": u, "v": v})

n = uv.sizes["time"]
BASE = np.datetime64("1900-01-01T00:00:00")

time_dt = BASE + da.arange(n, chunks=100_000).astype("timedelta64[h]")
uv = uv.assign_coords(time=("time", time_dt))

In [3]:
data = uv.sel(level=250, time="2020-01-01T00:00:00").compute()


In [ ]:
out_path = "../data/sandy2012-windUV.nc"


data.to_netcdf(out_path)
print("wrote", out_path)


wrote ../data/uv_250hPa_2020-01-01T00.nc


In [17]:
ds = xr.open_dataset(out_path)
ds


<xarray.Dataset> Size: 8MB
Dimensions:    (latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    level      int64 8B ...
    time       datetime64[ns] 8B ...
Data variables:
    u          (latitude, longitude) float32 4MB ...
    v          (latitude, longitude) float32 4MB ...

In [35]:
# levels: 250, 300, ..., 850
levels = np.arange(250, 851, 50)
# levels = [250,500,850]

BASE = np.datetime64("1900-01-01T00:00:00")
start = np.datetime64("2012-10-22T00:00:00")
end   = np.datetime64("2012-11-02T23:00:00")

# convert datetimes -> integer hour indices since BASE
i0 = int((start - BASE) / np.timedelta64(1, "h"))
i1 = int((end   - BASE) / np.timedelta64(1, "h"))

slice(i0, i1 + 1)

subset = (
    uv
    .isel(time=slice(i0, i1 + 1))     # inclusive end hour
    .sel(level=levels)               # select 250..850 every 50
    .chunk({"time": 48, "level": len(levels)})
    # .chunk({"time": 1})
)

# read_opt = subset.chunk({"time": 24})
# write_opt = read_opt.chunk({"time": 1})
# with ProgressBar():
#     data = subset.compute()

In [ ]:
with ProgressBar():
    subset.to_zarr("../data/sandy2012-windUV_all_levels_from_250_to_850.zarr", mode="a", compute=True)
    
# with ProgressBar():
#     subset.to_netcdf("../data/sandy2012-windUV.nc", compute=True)
    
print("wrote", out_path)


/home/dmmsp/anaconda3/envs/Hurricane-Track-Explainer/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


[########################################] | 100% Completed | 2hr 33m
wrote ../data/katrina2012-windUV.nc


In [ ]:
ds = xr.open_dataset("../data/sandy2012-windUV_all_levels_from_250_to_850.zarr")
ds.sel(level=250, time="2012-10-28T00:00:00").load()

In [ ]:
ds = xr.open_dataset("../data/sandy2012-windUV_250_500_850.zarr")
ds.sel(level=250, time="2012-10-28T00:00:00").load()

<xarray.Dataset> Size: 8MB
Dimensions:    (latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    level      int64 8B 250
    time       datetime64[ns] 8B 2012-10-28
Data variables:
    u          (latitude, longitude) float32 4MB -5.341e-05 ... -5.341e-05
    v          (latitude, longitude) float32 4MB 0.001888 0.001888 ... 0.001888

In [ ]:
src_path = "../data/sandy2012-windUV_all_levels_from_250_to_850.zarr"
dst_path = "../data/sandy2012-windUV_all_levels_from_250_to_850-rechunked.zarr"

ds = xr.open_zarr(src_path, consolidated=False)

# target_chunks = {"time": 1, "level": 1, "latitude": 721, "longitude": 1440}
target_chunks = (1, 1, 721, 1440)

u = ds["u"].data
v = ds["v"].data

u2 = da.rechunk(u, target_chunks)
v2 = da.rechunk(v, target_chunks)

ds2 = ds.copy()
ds2["u"] = xr.DataArray(u2, dims=ds["u"].dims, coords=ds["u"].coords, attrs=ds["u"].attrs)
ds2["v"] = xr.DataArray(v2, dims=ds["v"].dims, coords=ds["v"].coords, attrs=ds["v"].attrs)

with ProgressBar():
    ds2.to_zarr(dst_path, mode="w", consolidated=False, compute=True)

[#############                           ] | 33% Completed | 36m 13ss

IOStream.flush timed out


[#############                           ] | 33% Completed | 37m 11s

IOStream.flush timed out


[##########################              ] | 66% Completed | 66m 50s

...

[##########################              ] | 66% Completed | 69m 11s

IOStream.flush timed out


[##########################              ] | 66% Completed | 71m 0ss

IOStream.flush timed out


[########################################] | 100% Completed | 85m 52s


In [ ]:
ds = xr.open_zarr('../data/sandy2012-windUV_all_levels_from_250_to_850-rechunked.zarr', consolidated=False)
print(ds.chunks)  # if this is None, you’re not dask-lazy


Frozen({'time': (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), 'level': (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), 'latitude': (721,), 'longitude': (1440,)})
